Proposition 1: People who are BOTH employees and individual customers

In [2]:
USE AdventureWorks2019;
GO

WITH Emp AS (
  SELECT e.BusinessEntityID
  FROM HumanResources.Employee AS e
),
IndivCust AS (
  SELECT c.PersonID AS BusinessEntityID       
  FROM Sales.Customer AS c
  WHERE c.PersonID IS NOT NULL
)
SELECT p.BusinessEntityID,
       p.FirstName,
       p.LastName
FROM Person.Person AS p
JOIN (
    SELECT BusinessEntityID FROM Emp
    INTERSECT
    SELECT BusinessEntityID FROM IndivCust
) AS bothSides
  ON p.BusinessEntityID = bothSides.BusinessEntityID
ORDER BY p.LastName, p.FirstName;


Commands completed successfully.

(0 rows affected)

Total execution time: 00:00:00.077

BusinessEntityID,FirstName,LastName


Proposition 2: Employees who are not individual customer

In [3]:
USE AdventureWorks2019;
GO

WITH Emp AS (
  SELECT e.BusinessEntityID
  FROM HumanResources.Employee AS e
),
IndivCust AS (
  SELECT c.PersonID AS BusinessEntityID
  FROM Sales.Customer AS c
  WHERE c.PersonID IS NOT NULL
)
SELECT p.BusinessEntityID,
       p.FirstName,
       p.LastName
FROM Person.Person AS p
JOIN (
    SELECT BusinessEntityID FROM Emp
    EXCEPT
    SELECT BusinessEntityID FROM IndivCust
) AS empOnly
  ON p.BusinessEntityID = empOnly.BusinessEntityID
ORDER BY p.LastName, p.FirstName;


Commands completed successfully.

(290 rows affected)

Total execution time: 00:00:00.073

BusinessEntityID,FirstName,LastName
285,Syed,Abbas
38,Kim,Abercrombie
211,Hazem,Abolrous
121,Pilar,Ackerman
67,Jay,Adams
270,François,Ajenstat
287,Amy,Alberts
207,Greg,Alderson
216,Sean,Alexander
227,Gary,Altman


Proposition 3: Territory that has at least one customer or at least one salesperson assigned

In [4]:
USE AdventureWorks2019;
GO

WITH CustTerr AS (
    SELECT DISTINCT c.TerritoryID
    FROM Sales.Customer AS c
    WHERE c.TerritoryID IS NOT NULL
),
SalesTerr AS (
    SELECT DISTINCT sp.TerritoryID
    FROM Sales.SalesPerson AS sp
    WHERE sp.TerritoryID IS NOT NULL
)
SELECT t.TerritoryID,
       t.Name       AS TerritoryName,
       t.[Group]    AS TerritoryGroup
FROM Sales.SalesTerritory AS t
JOIN (
      SELECT TerritoryID FROM CustTerr
      UNION                          -- distinct union of territories
      SELECT TerritoryID FROM SalesTerr
) AS u
  ON t.TerritoryID = u.TerritoryID
ORDER BY t.[Group], t.Name;


Commands completed successfully.

(10 rows affected)

Total execution time: 00:00:00.036

TerritoryID,TerritoryName,TerritoryGroup
7,France,Europe
8,Germany,Europe
10,United Kingdom,Europe
6,Canada,North America
3,Central,North America
2,Northeast,North America
1,Northwest,North America
5,Southeast,North America
4,Southwest,North America
9,Australia,Pacific


Proposition 4: Products that are currently sellable but do not appear in any sales order

In [5]:
USE AdventureWorks2019;
GO

WITH AllSellable AS (
    SELECT p.ProductID
    FROM Production.Product AS p
    WHERE p.SellEndDate IS NULL              -- currently for sale
),
EverOrdered AS (
    SELECT DISTINCT sod.ProductID
    FROM Sales.SalesOrderDetail AS sod
)
SELECT p.ProductID,
       p.ProductNumber,
       p.Name,
       p.ListPrice
FROM Production.Product AS p
JOIN (
    SELECT ProductID FROM AllSellable
    EXCEPT
    SELECT ProductID FROM EverOrdered
) AS neverOrdered
  ON p.ProductID = neverOrdered.ProductID
ORDER BY p.Name;


Commands completed successfully.

(223 rows affected)

Total execution time: 00:00:00.099

ProductID,ProductNumber,Name,ListPrice
1,AR-5381,Adjustable Race,0.00
3,BE-2349,BB Ball Bearing,0.00
2,BA-8327,Bearing Ball,0.00
316,BL-2036,Blade,0.00
324,CS-2812,Chain Stays,0.00
322,CR-7833,Chainring,0.00
320,CB-2903,Chainring Bolts,0.00
321,CN-6137,Chainring Nut,0.00
505,RA-7490,Cone-Shaped Race,0.00
323,CR-9981,Crown Race,0.00


Proposition 5: Build a combined email list for employees + individual customers

In [6]:
USE AdventureWorks2019;
GO

WITH EmpEmails AS (
    SELECT e.BusinessEntityID,
           ea.EmailAddress
    FROM HumanResources.Employee AS e
    JOIN Person.EmailAddress AS ea
      ON ea.BusinessEntityID = e.BusinessEntityID
),
CustEmails AS (
    SELECT c.PersonID AS BusinessEntityID,
           ea.EmailAddress
    FROM Sales.Customer AS c
    JOIN Person.EmailAddress AS ea
      ON ea.BusinessEntityID = c.PersonID
    WHERE c.PersonID IS NOT NULL          -- only individual customers
),
Combined AS (
    SELECT 'Employee' AS Source, EmailAddress FROM EmpEmails
    UNION ALL
    SELECT 'IndividualCustomer' AS Source, EmailAddress FROM CustEmails
)
SELECT  Source,
        COUNT(*)                           AS TotalEmails,        -- counts with duplicates
        COUNT(DISTINCT EmailAddress)       AS DistinctEmails      -- unique addresses per source
FROM Combined
GROUP BY Source
ORDER BY Source;


Commands completed successfully.

(2 rows affected)

Total execution time: 00:00:00.166

Source,TotalEmails,DistinctEmails
Employee,290,290
IndividualCustomer,19119,19119


Proposition 6: Products sold both online and in-store

In [7]:
USE AdventureWorks2019;
GO

-- Products that appear in online and in-store orders
WITH OnlineProd AS (
    SELECT DISTINCT sod.ProductID
    FROM Sales.SalesOrderDetail AS sod
    JOIN Sales.SalesOrderHeader AS soh
      ON soh.SalesOrderID = sod.SalesOrderID
    WHERE soh.OnlineOrderFlag = 1
),
StoreProd AS (
    SELECT DISTINCT sod.ProductID
    FROM Sales.SalesOrderDetail AS sod
    JOIN Sales.SalesOrderHeader AS soh
      ON soh.SalesOrderID = sod.SalesOrderID
    WHERE soh.OnlineOrderFlag = 0
),
BothChannels AS (
    SELECT ProductID FROM OnlineProd
    INTERSECT
    SELECT ProductID FROM StoreProd
)
SELECT  p.ProductID,
        p.ProductNumber,
        p.Name
FROM Production.Product AS p
JOIN BothChannels AS b
  ON b.ProductID = p.ProductID
ORDER BY p.Name;


Commands completed successfully.

(114 rows affected)

Total execution time: 00:00:00.197

ProductID,ProductNumber,Name
712,CA-1098,AWC Logo Cap
877,CL-9009,Bike Wash - Dissolver
866,VE-C304-L,"Classic Vest, L"
865,VE-C304-M,"Classic Vest, M"
864,VE-C304-S,"Classic Vest, S"
860,GL-H102-L,"Half-Finger Gloves, L"
859,GL-H102-M,"Half-Finger Gloves, M"
858,GL-H102-S,"Half-Finger Gloves, S"
876,RA-H123,Hitch Rack - 4-Bike
880,HY-1023-70,Hydration Pack - 70 oz.


Proposition 7: Customers who ordered in 2012 or 2013

In [10]:
USE AdventureWorks2019;
GO

-- Distinct customers per year
WITH y2012 AS (
    SELECT DISTINCT soh.CustomerID
    FROM Sales.SalesOrderHeader AS soh
    WHERE YEAR(soh.OrderDate) = 2012
),
y2013 AS (
    SELECT DISTINCT soh.CustomerID
    FROM Sales.SalesOrderHeader AS soh
    WHERE YEAR(soh.OrderDate) = 2013
)

-- 1) Distinct customers across 2012 OR 2013 (UNION removes duplicates)
SELECT c.CustomerID,
       COALESCE(pp.FirstName + ' ' + pp.LastName, s.Name) AS CustomerName
FROM Sales.Customer AS c
JOIN (
    SELECT CustomerID FROM y2012
    UNION
    SELECT CustomerID FROM y2013
) AS u
  ON u.CustomerID = c.CustomerID
LEFT JOIN Person.Person AS pp
  ON pp.BusinessEntityID = c.PersonID
LEFT JOIN Sales.Store AS s
  ON s.BusinessEntityID = c.StoreID
ORDER BY CustomerName;


Commands completed successfully.

(12456 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.167

CustomerID,CustomerName
29943,A. Leonetti
28866,Aaron Adams
20075,Aaron Allen
12067,Aaron Bryant
28187,Aaron Chen
29675,Aaron Con
18695,Aaron Diaz
19692,Aaron Edwards
25415,Aaron Evans
14617,Aaron Flores


Proposition 8: Territories that have orders but no assigned salesperson

In [11]:
USE AdventureWorks2019;
GO

WITH TerrWithOrders AS (
    SELECT DISTINCT soh.TerritoryID
    FROM Sales.SalesOrderHeader AS soh
    WHERE soh.TerritoryID IS NOT NULL
),
TerrWithSalespeople AS (
    SELECT DISTINCT sp.TerritoryID
    FROM Sales.SalesPerson AS sp
    WHERE sp.TerritoryID IS NOT NULL
),
NoRepTerr AS (
    SELECT TerritoryID FROM TerrWithOrders
    EXCEPT
    SELECT TerritoryID FROM TerrWithSalespeople
)
SELECT t.TerritoryID,
       t.Name       AS TerritoryName,
       t.[Group]    AS TerritoryGroup,
       ord.OrderCount
FROM NoRepTerr AS n
JOIN Sales.SalesTerritory AS t
  ON t.TerritoryID = n.TerritoryID
CROSS APPLY (
    SELECT COUNT(*) AS OrderCount
    FROM Sales.SalesOrderHeader AS soh
    WHERE soh.TerritoryID = n.TerritoryID
) AS ord
ORDER BY t.[Group], t.Name;


Commands completed successfully.

(0 rows affected)

Total execution time: 00:00:00.043

TerritoryID,TerritoryName,TerritoryGroup,OrderCount


Proposition 9: People who are either employees or store contacts (but not both)

In [12]:
USE AdventureWorks2019;
GO

WITH Employees AS (
    SELECT e.BusinessEntityID
    FROM HumanResources.Employee AS e
),
StoreContacts AS (
    SELECT s.SalesPersonID AS BusinessEntityID
    FROM Sales.Store AS s
    WHERE s.SalesPersonID IS NOT NULL
)
SELECT p.BusinessEntityID,
       p.FirstName,
       p.LastName,
       'EmployeeOnly' AS Category
FROM Person.Person AS p
WHERE p.BusinessEntityID IN (
    SELECT BusinessEntityID FROM Employees
    EXCEPT
    SELECT BusinessEntityID FROM StoreContacts
)

UNION

SELECT p.BusinessEntityID,
       p.FirstName,
       p.LastName,
       'StoreContactOnly' AS Category
FROM Person.Person AS p
WHERE p.BusinessEntityID IN (
    SELECT BusinessEntityID FROM StoreContacts
    EXCEPT
    SELECT BusinessEntityID FROM Employees
)
ORDER BY Category, p.LastName, p.FirstName;


Commands completed successfully.

(277 rows affected)

Total execution time: 00:00:00.159

BusinessEntityID,FirstName,LastName,Category
285,Syed,Abbas,EmployeeOnly
38,Kim,Abercrombie,EmployeeOnly
211,Hazem,Abolrous,EmployeeOnly
121,Pilar,Ackerman,EmployeeOnly
67,Jay,Adams,EmployeeOnly
270,François,Ajenstat,EmployeeOnly
287,Amy,Alberts,EmployeeOnly
207,Greg,Alderson,EmployeeOnly
216,Sean,Alexander,EmployeeOnly
227,Gary,Altman,EmployeeOnly


Proposition 10: Business entities that are both vendors and stores (customers)

In [13]:
USE AdventureWorks2019;
GO

WITH VendorBE AS (
    SELECT v.BusinessEntityID
    FROM Purchasing.Vendor AS v
),
StoreBE AS (
    SELECT s.BusinessEntityID
    FROM Sales.Store AS s
)
, Both AS (
    SELECT BusinessEntityID FROM VendorBE
    INTERSECT
    SELECT BusinessEntityID FROM StoreBE
)
SELECT b.BusinessEntityID,
       v.Name AS VendorName,
       s.Name AS StoreName
FROM Both AS b
JOIN Purchasing.Vendor AS v
  ON v.BusinessEntityID = b.BusinessEntityID
JOIN Sales.Store AS s
  ON s.BusinessEntityID = b.BusinessEntityID
ORDER BY s.Name;


Commands completed successfully.

(0 rows affected)

Total execution time: 00:00:00.053

BusinessEntityID,VendorName,StoreName
